# Triton + PTX + PyTorch Kernel Primer — T4

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/HisameOgasahara/CS336_tmp/blob/main/triton_ptx_pytorch_primer_t4.ipynb)

이 노트북의 목적은 **리포지토리의 `cs336_gpu_kernels_triton_t4.ipynb`를 읽기 전에 필요한 최소 문법과 GPU kernel 감각을 직접 확인하는 것**이다.

진행 순서:

**PyTorch 연산 → GPU kernel 관점 → Triton 문법 → correctness → PTX → 자주 나는 에러**

끝나면 기존 노트북의 GeLU / Softmax / Row Sum / MatMul+ReLU / RMSNorm / FlashAttention-style 코드를 읽을 준비가 된다.

## 0. 가장 먼저 잡을 그림

CPU에서 Python 함수를 호출하는 것과 GPU kernel 실행은 다르다.

- **kernel**: GPU에서 병렬로 실행되는 함수
- **program**: Triton에서 같은 kernel의 한 작업 단위. CUDA의 block과 비슷한 역할
- **lane/thread**: Triton 코드는 thread를 직접 하나씩 쓰기보다 **벡터 block**을 기술한다
- **grid**: program을 몇 개 띄울지 정한다
- **HBM(VRAM)**: 큰 전역 메모리. 읽고 쓰는 비용이 비싸다
- **register/shared memory**: GPU 안쪽의 훨씬 빠른 저장공간
- 핵심 최적화 목표: **HBM 왕복을 줄이고, 한 번 읽은 데이터를 가능한 많이 재사용한다**

Stanford CS336의 systems 파트도 kernel의 핵심 원리를 “data movement 최소화”로 잡는다. fileciteturn0file7

In [1]:
import importlib.util
import subprocess
import sys

if importlib.util.find_spec("triton") is None:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "triton"])

import torch
import triton
import triton.language as tl

assert torch.cuda.is_available(), "Colab에서 Runtime > Change runtime type > T4 GPU를 선택하세요."

device = "cuda"

print("PyTorch:", torch.__version__)
print("Triton :", triton.__version__)
print("GPU    :", torch.cuda.get_device_name())
print("CC     :", torch.cuda.get_device_capability())

PyTorch: 2.11.0+cu128
Triton : 3.6.0
GPU    : Tesla T4
CC     : (7, 5)


## 1. PyTorch 한 줄도 결국 kernel을 실행한다

`y = x + 1`은 Python이 GPU의 원소를 직접 더하는 것이 아니다. PyTorch가 CUDA kernel을 launch한다.

아래 profiler에서 `aten::add`, `aten::relu` 같은 primitive가 각각 GPU kernel로 내려가는 모습을 본다.

In [2]:
from torch.profiler import profile, ProfilerActivity

x = torch.randn(1_000_000, device=device)

with profile(activities=[ProfilerActivity.CPU, ProfilerActivity.CUDA]) as p:
    y = torch.relu(x + 1.0)
    torch.cuda.synchronize()

print(p.key_averages().table(sort_by="cuda_time_total", row_limit=12))

-------------------------------------------------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  
                                                   Name    Self CPU %      Self CPU   CPU total %     CPU total  CPU time avg     Self CUDA   Self CUDA %    CUDA total  CUDA time avg    # of Calls  
-------------------------------------------------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  
                                       cudaLaunchKernel         0.95%     858.577us        70.47%      63.737ms      31.868ms       0.000us         0.00%     177.501us      88.750us             2  
                                             aten::relu         5.85%       5.291ms        49.96%      45.185ms      45.185ms       0.000us         0.00%     140.540us     140.540us             1  
         

/usr/local/lib/python3.13/dist-packages/torch/profiler/profiler.py:224: UserWarning: Warning: Profiler clears events at the end of each cycle.Only events from the current cycle will be reported.To keep events across cycles, set acc_events=True.
  _warn_once(


### 확인 포인트

PyTorch eager에서

```
x + 1
relu(...)
```

처럼 primitive가 둘이면 보통 **중간 tensor를 HBM에 쓰고 다시 읽는 kernel 2개**가 된다.

Triton을 배우는 첫 이유는 이 둘을 **한 kernel 안에서 fuse**할 수 있기 때문이다.

## 2. 가장 작은 Triton kernel: vector add

먼저 다음 타입을 명확히 구분한다.

- (x,y,z in mathbb{R}^N): GPU tensor
- `x_ptr, y_ptr, z_ptr`: 위 tensor의 시작 주소를 가리키는 GPU pointer
- `pid = tl.program_id(0) in mathbb{N}): 현재 program 번호
- `offsets in mathbb{N}^{B}): 현재 program이 맡는 원소 index 벡터
- `mask in \{\mathrm{False},\mathrm{True}\}^{B}`: 유효 index 여부
- (B in mathbb{N}): compile-time block size

목적은 (z_i=x_i+y_i)를 (i=0,dots,N-1)에 대해 병렬 계산하는 것이다.

In [3]:
@triton.jit
def add_kernel(x_ptr, y_ptr, z_ptr, n_elements: tl.constexpr, BLOCK_SIZE: tl.constexpr):
    pid = tl.program_id(axis=0)

    offsets = pid * BLOCK_SIZE + tl.arange(0, BLOCK_SIZE)
    mask = offsets < n_elements

    x = tl.load(x_ptr + offsets, mask=mask)
    y = tl.load(y_ptr + offsets, mask=mask)

    z = x + y

    tl.store(z_ptr + offsets, z, mask=mask)


def add_triton(x, y, block_size=256, return_handle=False):
    assert x.is_cuda and y.is_cuda
    assert x.shape == y.shape

    z = torch.empty_like(x)
    n = x.numel()

    grid = (triton.cdiv(n, block_size),)

    handle = add_kernel[grid](
        x, y, z, n,
        BLOCK_SIZE=block_size,
    )
    return (z, handle) if return_handle else z


x = torch.randn(1003, device=device)
y = torch.randn_like(x)

z, add_handle = add_triton(x, y, return_handle=True)
torch.testing.assert_close(z, x + y)

print("correct:", torch.allclose(z, x + y))
print("grid programs:", triton.cdiv(x.numel(), 256))

correct: True
grid programs: 4


### 이 6줄이 거의 모든 Triton kernel의 뼈대다

1. `tl.program_id` — 나는 몇 번째 program인가?
2. `tl.arange` — 내가 처리할 local index 묶음 생성
3. pointer arithmetic — `ptr + offsets`
4. `tl.load` — HBM → register
5. 계산
6. `tl.store` — register → HBM

기존 리포의 GeLU kernel도 사실 이 구조에 비선형 함수 하나를 넣은 것이다.

## 3. mask가 필요한 이유: 마지막 block은 보통 딱 맞지 않는다

(N=1003), (B=256)이면 program은 4개 필요하다. 마지막 program의 index는 768~1023이지만 실제 tensor는 1002까지만 존재한다.

그래서

```python
mask = offsets < n_elements
tl.load(..., mask=mask)
tl.store(..., mask=mask)
```

가 out-of-bounds 접근을 막는다.

In [4]:
N = 1003
B = 256
for pid in range(triton.cdiv(N, B)):
    start = pid * B
    end = start + B - 1
    valid = max(0, min(B, N - start))
    print(f"pid={pid}: logical [{start}, {end}], valid lanes={valid}")

pid=0: logical [0, 255], valid lanes=256
pid=1: logical [256, 511], valid lanes=256
pid=2: logical [512, 767], valid lanes=256
pid=3: logical [768, 1023], valid lanes=235


## 4. grid는 “GPU thread 수”가 아니라 “Triton program 수”

`kernel[grid](...)`가 launch 문법이다.

1D elementwise:
```python
grid = (ceil(N / B),)
```

2D matrix:
```python
grid = (num_row_blocks, num_col_blocks)
```

`tl.program_id(axis=0)`, `tl.program_id(axis=1)`로 각 축 번호를 읽는다.

In [5]:
@triton.jit
def fill_program_id_kernel(out_ptr, n: tl.constexpr, BLOCK_SIZE: tl.constexpr):
    pid = tl.program_id(0)
    offsets = pid * BLOCK_SIZE + tl.arange(0, BLOCK_SIZE)
    mask = offsets < n

    # 각 원소에 "어느 program이 처리했는지"를 기록
    value = tl.full((BLOCK_SIZE,), pid, tl.int32)
    tl.store(out_ptr + offsets, value, mask=mask)


out = torch.empty(20, device=device, dtype=torch.int32)
fill_program_id_kernel[(triton.cdiv(out.numel(), 8),)](
    out, out.numel(), BLOCK_SIZE=8
)
print(out.cpu().tolist())

[0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 1, 1, 1, 2, 2, 2, 2]


예상 형태:

```
[0,0,0,0,0,0,0,0, 1,1,1,1,1,1,1,1, 2,2,2,2]
```

즉 program 하나가 원소 하나가 아니라 **원소 묶음(block)**을 처리한다.

## 5. `tl.constexpr`: compile time에 값이 고정되는 인자

`BLOCK_SIZE`, tile 크기, head dimension처럼 **코드 생성 시 구조를 바꾸는 값**에 사용한다.

Triton compiler는 이 값을 알고 loop unroll, vectorization, register allocation 등을 결정할 수 있다.

반대로 runtime마다 달라지는 tensor 값 자체는 constexpr가 아니다.

In [6]:
@triton.jit
def scale_kernel(x_ptr, y_ptr, n, SCALE: tl.constexpr, BLOCK_SIZE: tl.constexpr):
    pid = tl.program_id(0)
    offsets = pid * BLOCK_SIZE + tl.arange(0, BLOCK_SIZE)
    mask = offsets < n
    x = tl.load(x_ptr + offsets, mask=mask)
    tl.store(y_ptr + offsets, x * SCALE, mask=mask)


x = torch.arange(10, device=device, dtype=torch.float32)
y = torch.empty_like(x)

scale_kernel[(1,)](x, y, x.numel(), SCALE=3.0, BLOCK_SIZE=16)
print(y.cpu())

tensor([ 0.,  3.,  6.,  9., 12., 15., 18., 21., 24., 27.])


## 6. pointer arithmetic와 stride — PyTorch tensor 구조와 직접 연결

2D tensor (Xinmathbb{R}^{M\times N})에 대해 PyTorch의 `stride()`는 index가 1 증가할 때 메모리에서 몇 element 이동하는지 뜻한다.

주소는

$$
\operatorname{addr}(X_{ij})
=
\operatorname{base}(X)
+
i\,s_0
+
j\,s_1
$$

형태다.

Triton의 `x_ptr + row * stride_row + col * stride_col`은 바로 이 식이다.

In [7]:
a = torch.arange(12, device=device).reshape(3, 4)
b = a.t()

print("a.shape :", a.shape, "a.stride:", a.stride(), "contiguous:", a.is_contiguous())
print("b.shape :", b.shape, "b.stride:", b.stride(), "contiguous:", b.is_contiguous())
print("a =\n", a.cpu())
print("b =\n", b.cpu())

a.shape : torch.Size([3, 4]) a.stride: (4, 1) contiguous: True
b.shape : torch.Size([4, 3]) b.stride: (1, 4) contiguous: False
a =
 tensor([[ 0,  1,  2,  3],
        [ 4,  5,  6,  7],
        [ 8,  9, 10, 11]])
b =
 tensor([[ 0,  4,  8],
        [ 1,  5,  9],
        [ 2,  6, 10],
        [ 3,  7, 11]])


## 7. reduction: `tl.sum`, `tl.max`

Softmax와 RMSNorm을 읽으려면 reduction이 핵심이다.

한 row (xinmathbb{R}^{N})의 합은

$$
s(x):=\sum_{j=1}^{N}x_j\in\mathbb{R}
$$

이다.

Triton에서는 한 program이 row를 register로 읽고 `tl.sum(x, axis=0)`로 줄일 수 있다.

In [8]:
@triton.jit
def row_sum_kernel(x_ptr, out_ptr, n_cols: tl.constexpr, BLOCK_SIZE: tl.constexpr):
    row = tl.program_id(0)

    cols = tl.arange(0, BLOCK_SIZE)
    mask = cols < n_cols

    x = tl.load(x_ptr + row * n_cols + cols, mask=mask, other=0.0)
    s = tl.sum(x, axis=0)

    tl.store(out_ptr + row, s)


x = torch.randn(5, 1000, device=device)
out = torch.empty(5, device=device)

row_sum_kernel[(x.shape[0],)](
    x, out, x.shape[1],
    BLOCK_SIZE=triton.next_power_of_2(x.shape[1]),
)

torch.testing.assert_close(out, x.sum(dim=1), rtol=1e-4, atol=1e-4)
print(out)
print(x.sum(dim=1))

tensor([-9.4417e-01,  8.6659e+00, -1.2263e+01,  7.4328e+01,  4.7281e-02],
       device='cuda:0')
tensor([-9.4417e-01,  8.6659e+00, -1.2263e+01,  7.4328e+01,  4.7281e-02],
       device='cuda:0')


### 왜 `next_power_of_2`를 쓰나?

Triton의 `tl.arange(0, BLOCK_SIZE)` block 크기는 compiler 제약 때문에 보통 2의 거듭제곱을 쓴다.

(N=1000)이면 (B=1024)로 잡고 뒤 24개 lane은 mask로 버린다.

## 8. Softmax: reduction + elementwise + fusion

row (xinmathbb{R}^{N})에 대해 numerical stability를 위한 softmax는

$$
m:=\max_j x_j,\qquad
p_i:=\frac{e^{x_i-m}}{\sum_j e^{x_j-m}}
$$

이다.

이 식을 한 program 안에서 끝내면 max, exp, sum, divide 사이의 중간 tensor를 HBM에 쓸 필요가 없다.

In [9]:
@triton.jit
def softmax_kernel(x_ptr, y_ptr, n_cols: tl.constexpr, BLOCK_SIZE: tl.constexpr):
    row = tl.program_id(0)

    cols = tl.arange(0, BLOCK_SIZE)
    mask = cols < n_cols

    x = tl.load(
        x_ptr + row * n_cols + cols,
        mask=mask,
        other=-float("inf"),
    )

    x = x - tl.max(x, axis=0)
    numerator = tl.exp(x)
    denominator = tl.sum(numerator, axis=0)
    y = numerator / denominator

    tl.store(y_ptr + row * n_cols + cols, y, mask=mask)


x = torch.randn(128, 1000, device=device)
y = torch.empty_like(x)

softmax_kernel[(x.shape[0],)](
    x, y, x.shape[1],
    BLOCK_SIZE=triton.next_power_of_2(x.shape[1]),
)

torch.testing.assert_close(y, torch.softmax(x, dim=1), rtol=1e-4, atol=1e-5)
print("max abs error:", (y - torch.softmax(x, dim=1)).abs().max().item())

max abs error: 7.450580596923828e-09


## 9. RMSNorm을 읽기 위한 PyTorch → Triton 대응

입력 (X\in\mathbb{R}^{M\times N}), weight (w\in\mathbb{R}^{N}), (arepsilon>0)에 대해 row (i)의 RMSNorm은

$$
r_i:=\left(\frac1N\sum_{j=1}^{N}X_{ij}^2+\varepsilon\right)^{-1/2},
\qquad
Y_{ij}:=X_{ij}\,r_i\,w_j.
$$

필요한 연산은:

`load → square → sum reduction → rsqrt → multiply → store`

즉 **row-wise reduction + elementwise fusion**이다. 기존 리포의 fused RMSNorm이 바로 이 패턴이다.

In [10]:
def rmsnorm_torch(x, weight, eps=1e-6):
    inv_rms = torch.rsqrt(x.float().pow(2).mean(dim=-1, keepdim=True) + eps)
    return (x.float() * inv_rms * weight.float()).to(x.dtype)

x = torch.randn(4, 8, device=device, dtype=torch.float16)
w = torch.randn(8, device=device, dtype=torch.float16)
print(rmsnorm_torch(x, w))

tensor([[-2.3418, -0.3044,  0.2072, -1.3037,  0.6382, -0.1824, -1.7998, -0.1256],
        [ 3.0586, -0.5229,  0.1141, -1.1494,  0.8174, -0.9277,  0.0593,  0.0522],
        [ 0.2426,  0.2583, -0.1312, -0.5361, -1.5117,  0.9941,  2.0820, -0.7329],
        [ 2.2461,  0.0518, -0.1631,  1.0596, -0.2539, -0.4028, -2.1816,  0.1350]],
       device='cuda:0', dtype=torch.float16)


## 10. MatMul을 읽기 위한 핵심: tile과 재사용

(A\in\mathbb{R}^{M\times K}), (B\in\mathbb{R}^{K\times N})에 대해

$$
C_{ij}:=\sum_{k=1}^{K}A_{ik}B_{kj}
$$

를 계산한다.

elementwise와 달리 한 (A_{ik})는 여러 (j)에서, 한 (B_{kj})는 여러 (i)에서 재사용된다.

그래서 matrix multiplication kernel은 전체 행렬을 한 번에 보지 않고

- (A)의 (B_M\times B_K) tile
- (B)의 (B_K\times B_N) tile

을 읽어 register/shared memory에 두고 `tl.dot`으로 누적한다.

In [11]:
@triton.jit
def tiny_matmul_kernel(
    a_ptr, b_ptr, c_ptr,
    M: tl.constexpr, N: tl.constexpr, K: tl.constexpr,
    BLOCK_M: tl.constexpr, BLOCK_N: tl.constexpr, BLOCK_K: tl.constexpr,
):
    pid_m = tl.program_id(0)
    pid_n = tl.program_id(1)

    offs_m = pid_m * BLOCK_M + tl.arange(0, BLOCK_M)
    offs_n = pid_n * BLOCK_N + tl.arange(0, BLOCK_N)
    offs_k = tl.arange(0, BLOCK_K)

    acc = tl.zeros((BLOCK_M, BLOCK_N), dtype=tl.float32)

    for k_start in range(0, K, BLOCK_K):
        k = k_start + offs_k

        a_ptrs = a_ptr + offs_m[:, None] * K + k[None, :]
        b_ptrs = b_ptr + k[:, None] * N + offs_n[None, :]

        a = tl.load(a_ptrs, mask=(offs_m[:, None] < M) & (k[None, :] < K), other=0.0)
        b = tl.load(b_ptrs, mask=(k[:, None] < K) & (offs_n[None, :] < N), other=0.0)

        acc += tl.dot(a, b)

    c_ptrs = c_ptr + offs_m[:, None] * N + offs_n[None, :]
    tl.store(
        c_ptrs,
        acc,
        mask=(offs_m[:, None] < M) & (offs_n[None, :] < N),
    )


M = N = K = 64
a = torch.randn(M, K, device=device, dtype=torch.float16)
b = torch.randn(K, N, device=device, dtype=torch.float16)
c = torch.empty(M, N, device=device, dtype=torch.float32)

tiny_matmul_kernel[(triton.cdiv(M, 16), triton.cdiv(N, 16))](
    a, b, c,
    M=M, N=N, K=K,
    BLOCK_M=16, BLOCK_N=16, BLOCK_K=16,
    num_warps=4,
)

torch.testing.assert_close(c, a.float() @ b.float(), rtol=2e-2, atol=2e-2)
print("max abs error:", (c - a.float() @ b.float()).abs().max().item())

max abs error: 7.62939453125e-06


## 11. PyTorch 주요 구조를 Triton 관점으로 번역

| PyTorch 구조 | Triton에서 보는 핵심 |
|---|---|
| `x + y`, activation | elementwise, 1D block |
| `sum/max/mean(dim=...)` | reduction |
| LayerNorm / RMSNorm | row reduction + elementwise fusion |
| Linear / MatMul | 2D tiling + `tl.dot` |
| MatMul + bias + activation | matmul epilogue fusion |
| Softmax | max/sum reduction + fusion |
| Attention | tiled QKᵀ + online softmax + tiled PV |
| transpose/view | 실제 copy인지 stride 변화인지 구분 |

이 표가 기존 실습 노트북의 구조를 읽는 지도다.

## 12. PTX는 왜 보나?

흐름은 대략

```
Triton Python
  ↓
Triton IR / LLVM
  ↓
PTX
  ↓
SASS (실제 GPU 명령)
```

이다.

PTX에서 처음에는 다 읽지 말고 다음만 찾는다.

- `.target sm_75` : T4의 compute capability 7.5
- `%ctaid.x` : block/program ID 계열
- `%tid.x` : thread ID
- `ld.global` : HBM/global memory load
- `st.global` : HBM/global memory store
- `.reg` : register 선언
- `bar.sync` : synchronization
- `mma` 계열 : tensor core matrix multiply 관련 명령

목적은 assembly 공부 자체가 아니라 **내 Triton 코드가 실제로 어떤 memory access와 계산으로 내려갔는지 확인**하는 것이다.

In [12]:
def show_ptx(handle, keywords=None, n=120):
    ptx = handle.asm["ptx"].splitlines()
    if keywords is not None:
        ptx = [line for line in ptx if any(k in line for k in keywords)]
    print("\n".join(ptx[:n]))

print("=== PTX header / first lines ===")
show_ptx(add_handle, n=40)

print("\n=== memory / ids / registers only ===")
show_ptx(
    add_handle,
    keywords=[".target", ".reg", "ld.global", "st.global", "%ctaid", "%tid"],
    n=80,
)

=== PTX header / first lines ===
//
// Generated by LLVM NVPTX Back-End
//

.version 8.7
.target sm_75
.address_size 64

	// .globl	add_kernel              // -- Begin function add_kernel
                                        // @add_kernel
.visible .entry add_kernel(
	.param .u64 .ptr .global .align 1 add_kernel_param_0,
	.param .u64 .ptr .global .align 1 add_kernel_param_1,
	.param .u64 .ptr .global .align 1 add_kernel_param_2,
	.param .u64 .ptr .global .align 1 add_kernel_param_3,
	.param .u64 .ptr .global .align 1 add_kernel_param_4
)
.reqntid 128
{
	.reg .pred 	%p<3>;
	.reg .b32 	%r<14>;
	.reg .b64 	%rd<11>;
	.loc	1 2 0                           // 649003326.py:2:0
$L__func_begin0:
	.loc	1 2 0                           // 649003326.py:2:0

// %bb.0:
	ld.param.b64 	%rd7, [add_kernel_param_0];
	ld.param.b64 	%rd8, [add_kernel_param_1];
$L__tmp0:
	.loc	1 3 24                          // 649003326.py:3:24
	mov.u32 	%r7, %ctaid.x;
	.loc	1 5 20                          // 649003326.py

## 13. benchmark: kernel은 반드시 synchronize해서 잰다

GPU 실행은 asynchronous라서 Python의 호출 종료 시점이 GPU 계산 종료 시점과 같지 않다.

그래서 CUDA Event 또는 Triton benchmark helper를 사용한다.

In [13]:
def bench_ms(fn, warmup=20, trials=100):
    for _ in range(warmup):
        fn()
    torch.cuda.synchronize()

    start = torch.cuda.Event(enable_timing=True)
    end = torch.cuda.Event(enable_timing=True)

    start.record()
    for _ in range(trials):
        fn()
    end.record()

    torch.cuda.synchronize()
    return start.elapsed_time(end) / trials


x = torch.randn(4_000_000, device=device)
y = torch.randn_like(x)

print("torch add :", bench_ms(lambda: x + y), "ms")
print("triton add:", bench_ms(lambda: add_triton(x, y)), "ms")

torch add : 0.19924671173095704 ms
triton add: 0.18825471878051758 ms


## 14. Error Lab — 일부러 틀려 보고 에러 메시지 읽기

아래 셀들은 notebook 실행을 중단하지 않도록 `try/except`로 감싼다.

에러 메시지는 Triton/PyTorch 버전에 따라 조금 달라질 수 있으니 **문구 암기보다 원인 분류**가 목적이다.

### Error 1 — CPU tensor를 Triton kernel에 넘김

Triton GPU kernel의 pointer 인자는 GPU memory를 가리켜야 한다.

In [14]:
x_cpu = torch.randn(16)
y_cpu = torch.randn(16)
z_cpu = torch.empty_like(x_cpu)

try:
    add_kernel[(1,)](
        x_cpu, y_cpu, z_cpu, 16,
        BLOCK_SIZE=16,
    )
except Exception as e:
    print(type(e).__name__ + ":")
    print(e)

ValueError:
Pointer argument (at 0) cannot be accessed from Triton (cpu tensor?)


대표적으로 **pointer argument가 GPU에서 접근 가능하지 않다**는 종류의 메시지가 나온다.

해결: input/output tensor의 `.device`를 확인하고 CUDA tensor로 맞춘다.

### Error 2 — `tl.arange` block size를 엉뚱하게 줌

Triton block shape에는 compiler 제약이 있다. 특히 reduction/벡터 block은 2의 거듭제곱 크기를 습관적으로 쓰는 편이 안전하다.

In [15]:
@triton.jit
def bad_arange_kernel(x_ptr):
    offsets = tl.arange(0, 1000)
    x = tl.load(x_ptr + offsets)
    tl.store(x_ptr + offsets, x)

x = torch.zeros(1024, device=device)

try:
    bad_arange_kernel[(1,)](x)
except Exception as e:
    print(type(e).__name__ + ":")
    print(e)

CompilationError:
at 2:14:
def bad_arange_kernel(x_ptr):
    offsets = tl.arange(0, 1000)
              ^
arange's range must be a power of 2


대표 원인: `tl.arange` 범위/shape가 compiler가 요구하는 block 조건과 맞지 않음.

해결:

```python
BLOCK_SIZE = triton.next_power_of_2(n)
offsets = tl.arange(0, BLOCK_SIZE)
mask = offsets < n
```

### Error 3 — shape는 맞아 보이지만 stride/contiguous 가정을 깨뜨림

아래 kernel은 2D tensor가 row-major contiguous라고 **가정**한다. transpose tensor를 넣으면 compile error가 아니라 더 위험한 **조용한 오답**이 날 수 있다.

In [16]:
@triton.jit
def copy_assuming_contiguous_kernel(x_ptr, y_ptr, n: tl.constexpr, BLOCK_SIZE: tl.constexpr):
    pid = tl.program_id(0)
    offsets = pid * BLOCK_SIZE + tl.arange(0, BLOCK_SIZE)
    mask = offsets < n
    x = tl.load(x_ptr + offsets, mask=mask)
    tl.store(y_ptr + offsets, x, mask=mask)


a = torch.arange(12, device=device, dtype=torch.float32).reshape(3, 4)
b = a.t()  # non-contiguous view

out = torch.empty_like(b)

copy_assuming_contiguous_kernel[(1,)](
    b, out, b.numel(),
    BLOCK_SIZE=16,
)

print("input stride :", b.stride())
print("equal?       :", torch.equal(out, b))
print("expected:\n", b.cpu())
print("got:\n", out.cpu())

input stride : (1, 4)
equal?       : True
expected:
 tensor([[ 0.,  4.,  8.],
        [ 1.,  5.,  9.],
        [ 2.,  6., 10.],
        [ 3.,  7., 11.]])
got:
 tensor([[ 0.,  4.,  8.],
        [ 1.,  5.,  9.],
        [ 2.,  6., 10.],
        [ 3.,  7., 11.]])


이게 실무에서 더 중요한 에러다. **kernel이 실행됐다고 correct한 것이 아니다.**

따라서 custom kernel마다 최소한

```python
torch.testing.assert_close(custom, reference)
```

를 먼저 통과시키고 benchmark를 해야 한다.

해결법은 두 가지다.

1. wrapper에서 `assert x.is_contiguous()`
2. kernel에 stride를 명시적으로 전달하여 실제 layout을 지원

### Error 4 — dtype / accumulation precision

FP16 reduction이나 matmul은 중간 누적을 FP32로 올리는 이유가 있다. 아래에서 오차 차이를 본다.

In [17]:
torch.manual_seed(0)

x = torch.randn(1_000_000, device=device, dtype=torch.float16) * 0.1

sum_fp16 = x.sum(dtype=torch.float16)
sum_fp32 = x.sum(dtype=torch.float32)
sum_ref = x.double().sum()

print("FP16 abs error:", float((sum_fp16.double() - sum_ref).abs()))
print("FP32 abs error:", float((sum_fp32.double() - sum_ref).abs()))

FP16 abs error: 0.011708259582519531
FP32 abs error: 2.86102294921875e-06


Triton 코드에서 reduction/matmul accumulator를 `tl.float32`로 두는 패턴을 자주 보는 이유다.

## 15. 기존 리포 코드를 읽는 체크리스트

이제 `cs336_gpu_kernels_triton_t4.ipynb`의 kernel마다 아래 순서로 읽으면 된다.

1. **입출력 tensor shape와 dtype은?**
2. **grid는 몇 차원이고 program 하나가 무엇을 맡나?**
3. **`tl.arange`가 만든 축은 무엇인가?**
4. **pointer arithmetic이 어떤 tensor index를 뜻하나?**
5. **mask가 어떤 경계를 막나?**
6. **reduction / `tl.dot` / elementwise 중 무엇인가?**
7. **어떤 중간 tensor를 HBM에 쓰지 않게 되었나?**
8. **`num_warps`, tile size를 바꾸면 register/occupancy가 어떻게 달라질까?**
9. **PyTorch reference와 correctness를 먼저 검증했나?**
10. **PTX에서 load/store/register가 예상대로 보이나?**

이 관점으로 기존 노트북의 흐름은:

**GeLU → Softmax → Row Sum → MatMul+ReLU → RMSNorm → FlashAttention-style**

순서로 난도가 올라간다.

## 16. 최종 미니 과제

아래 세 개만 직접 수정해보면 읽기 능력이 크게 올라간다.

1. `add_kernel`을 (z=\operatorname{ReLU}(x+y))로 바꾸고 PyTorch eager와 profiler 비교
2. `row_sum_kernel`을 row mean으로 바꾸기
3. `tiny_matmul_kernel`의 store 직전에 ReLU를 붙여 **MatMul+ReLU fusion** 만들기

성능보다 먼저 **왜 HBM write/read가 하나 줄었는지** 설명할 수 있으면 성공이다.